## `시도별 csv 3개(+신규 1개) 생성 하는 내용들입니다`

# 2단계: 시/도별 개선율 산정 + SAW 입력 데이터 생성

1단계 - 전국 도시대기 평균 기준 계절관리제 시행 이후 겨울철 PM10/PM2.5 농도가 감소했는지 확인
2단계 - 감소폭을 **시/도별로 분해**해서, 개선이 미흡한 지역을 찾고 SAW 우선순위 산정에 사용할 입력 데이터를 만든다.

**SAW 대안**: PM10 기준 17개 시/도, PM2.5 기준 15개 시/도 (전남·경북 제외)

**SAW 기준**
- PM2.5 개선율
- PM10 개선율
- 시행 후 PM2.5 절대농도

**우선순위 방향**
- 개선율은 낮을수록 개선이 미흡하므로 우선순위가 높다.
- 시행 후 PM2.5 절대농도는 높을수록 현재 오염 부담이 크므로 우선순위가 높다.

## `0~2 사전작업. `
## 0. 라이브러리 및 경로 설정

노트북 실행 위치가 프로젝트 루트일 수도 있고 `src` 폴더일 수도 있으므로, 두 가지 경로를 모두 후보로 둔다.
`processed_by_region/__init__.py`의 PM10_START, PM25_START, EXCLUDED_PM25_SIDO를 import해서 사용한다.

In [20]:
from pathlib import Path
import sys
import pandas as pd

# processed_by_region 패키지를 import할 수 있도록 sys.path에 추가한다.
# 실행 위치가 프로젝트 루트인 경우와 src 폴더인 경우를 모두 대응한다.
REGION_DIR_CANDIDATES = [
    Path("data/processed/processed_by_region"),
    Path("../data/processed/processed_by_region"),
]

OUTPUT_DIR_CANDIDATES = [
    Path("data/processed"),
    Path("../data/processed"),
]

BEFORE_LABEL = "시행 전 (2015~2018)"
AFTER_LABEL  = "시행 후 (2019~2023)"
WINTER_MONTHS = [12, 1, 2, 3]


def resolve_existing_dir(candidates: list[Path]) -> Path:
    """후보 경로 중 실제 존재하는 첫 번째 폴더를 반환한다."""
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"폴더를 찾을 수 없습니다: {candidates}")


def resolve_output_dir(candidates: list[Path]) -> Path:
    """후보 경로 중 실제 존재하는 첫 번째 출력 폴더를 반환한다."""
    for path in candidates:
        if path.exists():
            return path
    candidates[0].mkdir(parents=True, exist_ok=True)
    return candidates[0]


REGION_DIR = resolve_existing_dir(REGION_DIR_CANDIDATES)
OUTPUT_DIR = resolve_output_dir(OUTPUT_DIR_CANDIDATES)

# processed_by_region 패키지 경로를 sys.path에 추가해 PM10_START 등을 import한다.
sys.path.insert(0, str(REGION_DIR.parent))
from processed_by_region import PM10_START, PM25_START, EXCLUDED_PM25_SIDO  # type: ignore

print("지역 데이터 폴더:", REGION_DIR)
print("출력 폴더:", OUTPUT_DIR)
print("PM10 시/도 수:", len(PM10_START))
print("PM2.5 시/도 수:", len(PM25_START))
print("PM2.5 제외 시/도:", list(EXCLUDED_PM25_SIDO.keys()))

지역 데이터 폴더: ..\data\processed\processed_by_region
출력 폴더: ..\data\processed
PM10 시/도 수: 17
PM2.5 시/도 수: 15
PM2.5 제외 시/도: ['전남', '경북']


## 1. processed_by_region 에서 PM10 / PM2.5 데이터 읽기

`build_regional_parquet.py`가 이미 도시대기 필터링 + 결측치 제거 + 시/도별 분리까지 완료했다.
PM10(17개 시/도)과 PM2.5(15개 시/도)는 대상 지역이 다르므로 완전히 분리된 데이터프레임으로 로드한다.
각 시/도 parquet을 읽을 때 PM10_START / PM25_START에 명시된 유효 시작 연월 이전 데이터는 제거한다.

In [21]:
def load_region_parquet(
    region_dir: Path,
    start_dict: dict[str, str],
    pollutant: str,
) -> pd.DataFrame:
    """시/도별 parquet을 읽어 concat하고 유효 시작일 이전 데이터를 제거한다."""
    frames = []
    for sido, start_ym in start_dict.items():
        fpath = region_dir / f"{sido}_{pollutant}.parquet"
        df = pd.read_parquet(fpath)
        df["date"] = pd.to_datetime(df["date"])
        # 유효 시작일 이전 데이터는 결측이 많아 해석을 왜곡할 수 있으므로 제거한다.
        start_date = pd.Timestamp(f"{start_ym}-01")
        df = df[df["date"] >= start_date].copy()
        df["sido"] = sido
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


# network 필터링은 이미 완료되어 있으므로 별도로 필터링하지 않는다.
pm10_raw = load_region_parquet(REGION_DIR, PM10_START, "pm10")
pm25_raw = load_region_parquet(REGION_DIR, PM25_START, "pm25")

print("PM10 전체 데이터:", pm10_raw.shape)
print("PM10 시/도 목록:", sorted(pm10_raw["sido"].unique()))
print()
print("PM2.5 전체 데이터:", pm25_raw.shape)
print("PM2.5 시/도 목록:", sorted(pm25_raw["sido"].unique()))

PM10 전체 데이터: (1329571, 5)
PM10 시/도 목록: ['강원', '경기', '경남', '경북', '광주', '대구', '대전', '부산', '서울', '세종', '울산', '인천', '전남', '전북', '제주', '충남', '충북']

PM2.5 전체 데이터: (975684, 5)
PM2.5 시/도 목록: ['강원', '경기', '경남', '광주', '대구', '대전', '부산', '서울', '세종', '울산', '인천', '전북', '제주', '충남', '충북']


## 2. 겨울철 정의 및 시행 전후 그룹 구성

겨울철은 `12월~다음해 3월`로 정의한다.

예를 들어 `2019 겨울`은 `2019-12 ~ 2020-03`이다. 따라서 1~3월은 전년도 겨울로 귀속한다.

- 시행 전: 2015~2018 겨울
- 시행 후: 2019~2023 겨울

```2014 겨울과 2024 겨울은 잘려서 제외함.```

In [22]:
def assign_winter_period(df: pd.DataFrame, pollutant_col: str) -> pd.DataFrame:
    """겨울철(12~3월) 필터링 및 winter_year / period 컬럼을 부여한다."""
    w = df[df["date"].dt.month.isin(WINTER_MONTHS)].copy()
    # 12월은 그 해 겨울, 1~3월은 전년도 겨울로 묶는다.
    w["winter_year"] = w["date"].dt.year.where(
        w["date"].dt.month == 12,
        w["date"].dt.year - 1,
    )
    # 2014 겨울(데이터 불완전)과 2024 겨울(미완성)은 제외한다.
    w = w[w["winter_year"].between(2015, 2023)]
    # 계절관리제 최초 시행은 2019년 12월이므로 2019 겨울부터 시행 후로 본다.
    w["period"] = w["winter_year"].apply(
        lambda year: BEFORE_LABEL if year <= 2018 else AFTER_LABEL
    )
    return w


pm10_winter = assign_winter_period(pm10_raw, "pm10")
pm25_winter = assign_winter_period(pm25_raw, "pm25")

print("[PM10] winter_year × period 분포:")
print(pm10_winter.groupby(["winter_year", "period"]).size())
print()
print("[PM2.5] winter_year × period 분포:")
print(pm25_winter.groupby(["winter_year", "period"]).size())

[PM10] winter_year × period 분포:
winter_year  period          
2015         시행 전 (2015~2018)    26627
2016         시행 전 (2015~2018)    28600
2017         시행 전 (2015~2018)    30070
2018         시행 전 (2015~2018)    39566
2019         시행 후 (2019~2023)    47175
2020         시행 후 (2019~2023)    54648
2021         시행 후 (2019~2023)    58774
2022         시행 후 (2019~2023)    60937
2023         시행 후 (2019~2023)    61830
dtype: int64

[PM2.5] winter_year × period 분포:
winter_year  period          
2015         시행 전 (2015~2018)     7029
2016         시행 전 (2015~2018)     8765
2017         시행 전 (2015~2018)    14319
2018         시행 전 (2015~2018)    34410
2019         시행 후 (2019~2023)    39324
2020         시행 후 (2019~2023)    45666
2021         시행 후 (2019~2023)    48869
2022         시행 후 (2019~2023)    50440
2023         시행 후 (2019~2023)    51286
dtype: int64


## 3. PM2.5 before 구간 표본 부족 체크

PM2.5는 시작 시점이 늦은 지역(경기·충남 2018-04, 경남·울산·전북 2018-02 등)의 경우
"시행 전(2015~2018 겨울)" 구간에 유효 겨울이 1개 이하일 수 있다.
표본이 부족한 지역은 before 평균의 신뢰도가 낮으므로 해석 시 주의가 필요하다.

In [23]:
# 시/도별 before 구간의 유효 winter_year 개수를 확인한다.
before_years_per_sido = (
    pm25_winter[pm25_winter["period"] == BEFORE_LABEL]
    .groupby("sido")["winter_year"]
    .nunique()
    .rename("before_winter_count")
)

print("[PM2.5] 시/도별 before 구간 유효 겨울 수:")
print(before_years_per_sido.sort_values())
print()

# 유효 겨울이 1개 이하인 지역은 경고를 출력한다.
low_sample = before_years_per_sido[before_years_per_sido <= 1]
if not low_sample.empty:
    print("[경고] before 구간 유효 겨울이 1개 이하인 시/도 (해석 주의)")
    for sido, cnt in low_sample.items():
        print(f"   - {sido}: {cnt}개 겨울")
else:
    print("[OK] 모든 PM2.5 시/도의 before 구간 겨울 수가 2개 이상입니다.")

[PM2.5] 시/도별 before 구간 유효 겨울 수:
sido
경기    1
충남    1
대전    2
경남    2
인천    2
전북    2
울산    2
대구    3
세종    3
제주    3
강원    4
광주    4
서울    4
부산    4
충북    4
Name: before_winter_count, dtype: int64

[경고] before 구간 유효 겨울이 1개 이하인 시/도 (해석 주의)
   - 경기: 1개 겨울
   - 충남: 1개 겨울


## 4. 시/도별 일평균으로 축약

원자료는 측정소-일 단위다. 시/도별 개선율을 계산하기 전에 `시/도 × 날짜 × 시행 전후` 단위로 평균을 낸다.

이렇게 하면 각 시/도는 하루에 오염물질 값을 하나씩 갖게 된다. PM10과 PM2.5는 완전히 분리된 프레임으로 처리한다.

In [24]:
def aggregate_daily(winter_df: pd.DataFrame, pollutant_col: str) -> pd.DataFrame:
    """시/도 × 날짜 × 시행 전후 단위로 측정소 평균을 낸다."""
    return (
        winter_df.groupby(["sido", "date", "period", "winter_year"])[[pollutant_col]]
        .mean()
        .reset_index()
    )


pm10_daily = aggregate_daily(pm10_winter, "pm10")
pm25_daily = aggregate_daily(pm25_winter, "pm25")

print("PM10 시/도별 일평균 데이터:", pm10_daily.shape)
pm10_daily.head(3)

PM10 시/도별 일평균 데이터: (18527, 5)


,sido,date,period,winter_year,pm10
0,강원,2015-12-01,시행 전 (2015~2018),2015,65.658333
1,강원,2015-12-02,시행 전 (2015~2018),2015,54.685606
2,강원,2015-12-03,시행 전 (2015~2018),2015,24.091667


## 5. 시/도별 시행 전후 평균 및 개선율 계산

개선율 공식은 다음과 같다.

`개선율 = (시행 전 평균 - 시행 후 평균) / 시행 전 평균 × 100`

값이 클수록 시행 후 농도가 더 많이 감소했다는 뜻이다. 값이 낮으면 개선이 미흡한 지역으로 볼 수 있다.
PM10(17개)과 PM2.5(15개)를 각각 계산한 뒤 sido 기준 outer merge로 합친다.

In [25]:
def calc_improvement(daily_df: pd.DataFrame, pollutant_col: str) -> pd.DataFrame:
    """시/도별 시행 전후 평균과 개선율을 계산한다."""
    period_mean = (
        daily_df.groupby(["sido", "period"])[[pollutant_col]]
        .mean()
        .unstack("period")
    )
    # unstack 후 멀티인덱스를 단순 컬럼으로 변환한다.
    # BEFORE_LABEL / AFTER_LABEL 키를 그대로 참조하므로 라벨이 바뀌어도 안전하다.
    before_vals = period_mean[(pollutant_col, BEFORE_LABEL)]
    after_vals  = period_mean[(pollutant_col, AFTER_LABEL)]
    result = pd.DataFrame({
        "sido":                     period_mean.index,
        f"{pollutant_col}_before":  before_vals.values,
        f"{pollutant_col}_after":   after_vals.values,
    })
    result[f"{pollutant_col}_improvement_rate"] = (
        (result[f"{pollutant_col}_before"] - result[f"{pollutant_col}_after"])
        / result[f"{pollutant_col}_before"]
        * 100
    )
    return result


pm10_impr = calc_improvement(pm10_daily, "pm10")
pm25_impr = calc_improvement(pm25_daily, "pm25")

# PM10(17개)과 PM2.5(15개)를 sido 기준 outer merge한다.
# 전남·경북은 PM2.5 관련 컬럼이 자연스럽게 NaN으로 남는다.
improvement = pd.merge(pm10_impr, pm25_impr, on="sido", how="outer")

# PM2.5 개선율 기준으로 정렬하되, NaN은 마지막에 온다 (전남·경북).
improvement = improvement.sort_values("pm25_improvement_rate", na_position="last").reset_index(drop=True)

print("improvement 테이블:", improvement.shape)
improvement.round(2)

improvement 테이블: (17, 7)


,sido,pm10_before,pm10_after,pm10_improvement_rate,pm25_before,pm25_after,pm25_improvement_rate
0,세종,52.31,46.28,11.54,29.82,26.07,12.55
1,인천,52.57,44.06,16.18,30.54,25.19,17.52
2,서울,53.53,44.03,17.75,31.95,25.61,19.83
3,대구,48.71,40.18,17.52,29.03,22.93,21.01
4,충남,51.51,47.35,8.06,35.16,27.62,21.45
5,경남,46.03,32.32,29.79,23.77,18.21,23.40
6,울산,43.11,34.86,19.14,24.68,18.64,24.50
7,광주,47.82,37.91,20.71,29.94,22.25,25.69
8,충북,55.17,44.88,18.65,36.81,27.26,25.94
9,제주,39.96,33.71,15.63,22.33,16.52,26.00


## 6. 개선 미흡 지역 확인

PM2.5 개선율 기준으로 하위 지역을 먼저 확인한다. 이 지역들은 SAW에서 높은 우선순위를 받을 가능성이 크다.
전남·경북은 PM2.5 분석에서 제외되므로 해당 행의 PM2.5 컬럼은 NaN으로 표시된다.

In [26]:
# PM2.5 개선율 하위 5개 지역 (NaN은 제외하고 정렬)
pm25_ranked = improvement.dropna(subset=["pm25_improvement_rate"])
print("PM2.5 개선율 하위 5개 시/도:")
print(pm25_ranked.head(5)[["sido", "pm25_before", "pm25_after", "pm25_improvement_rate"]].round(2))
print()
print("전남·경북 확인 (PM2.5 NaN 여부):")
print(improvement[improvement["sido"].isin(["전남", "경북"])][["sido", "pm10_improvement_rate", "pm25_improvement_rate"]].round(2))

PM2.5 개선율 하위 5개 시/도:
  sido  pm25_before  pm25_after  pm25_improvement_rate
0   세종        29.82       26.07                  12.55
1   인천        30.54       25.19                  17.52
2   서울        31.95       25.61                  19.83
3   대구        29.03       22.93                  21.01
4   충남        35.16       27.62                  21.45

전남·경북 확인 (PM2.5 NaN 여부):
   sido  pm10_improvement_rate  pm25_improvement_rate
15   경북                  19.00                    NaN
16   전남                  11.39                    NaN


## 7. 편향 점검표 생성

processed_by_region 데이터는 이미 결측치 제거가 완료된 상태이므로 결측률을 다시 계산하는 것은 의미가 없다.
대신 시/도별 측정소 수, 관측 일수, PM2.5 분석 시작 연도를 정리한 표로 편향 가능성을 점검한다.

In [27]:
# PM10: 시/도별 측정소 수 + 관측 일수
pm10_bias = (
    pm10_winter.groupby("sido")
    .agg(
        pm10_station_count=("station_code", "nunique"),
        pm10_obs_days=("date", "nunique"),
    )
    .reset_index()
)

# PM2.5: 시/도별 측정소 수 + 관측 일수 + PM2.5 분석 시작 연도
pm25_bias = (
    pm25_winter.groupby("sido")
    .agg(
        pm25_station_count=("station_code", "nunique"),
        pm25_obs_days=("date", "nunique"),
    )
    .reset_index()
)
# PM2.5 유효 시작 연도를 PM25_START에서 가져온다.
pm25_bias["pm25_start_year"] = pm25_bias["sido"].map(
    {sido: int(ym[:4]) for sido, ym in PM25_START.items()}
)

# outer merge: 전남·경북은 PM2.5 컬럼이 NaN으로 남는다.
bias = pd.merge(pm10_bias, pm25_bias, on="sido", how="outer")
# 전남·경북은 PM2.5 제외 시/도임을 명시한다.
bias["pm25_excluded"] = bias["sido"].isin(EXCLUDED_PM25_SIDO.keys())

bias = bias.sort_values("sido").reset_index(drop=True)
print("편향 점검표:", bias.shape)
bias

편향 점검표: (17, 7)


,sido,pm10_station_count,pm10_obs_days,pm25_station_count,pm25_obs_days,pm25_start_year,pm25_excluded
0,강원,24,1092,24.0,1059.0,2016.0,False
1,경기,109,1092,109.0,728.0,2018.0,False
2,경남,39,1092,39.0,787.0,2018.0,False
3,경북,49,1092,NaN,NaN,NaN,True
4,광주,11,1092,11.0,1092.0,2015.0,False
5,대구,20,1092,20.0,939.0,2017.0,False
6,대전,11,1092,11.0,849.0,2017.0,False
7,부산,27,1092,27.0,1092.0,2015.0,False
8,서울,25,1092,25.0,1092.0,2015.0,False
9,세종,4,1055,4.0,954.0,2016.0,False


## 8. 2017~2019 구간 vs 2020~ 구간 비교

2017년 이전 데이터는 결측치가 커 해석이 왜곡될 수 있다는 팀원 지적에 따라,
`winter_year` 기준 **2017~2019 구간**과 **2020년 이후 구간**을 별도로 비교한다.
결과는 `sido_period_compare_2017_2020.csv`로 저장한다.

In [28]:
def compare_two_periods(
    daily_df: pd.DataFrame,
    pollutant_col: str,
    early_range: tuple[int, int],
    late_range: tuple[int, int],
) -> pd.DataFrame:
    """두 winter_year 구간의 시/도별 평균과 변화율을 계산한다."""
    early = daily_df[daily_df["winter_year"].between(*early_range)]
    late  = daily_df[daily_df["winter_year"].between(*late_range)]
    early_mean = early.groupby("sido")[pollutant_col].mean().rename(f"{pollutant_col}_2017_2019_mean")
    late_mean  = late.groupby("sido")[pollutant_col].mean().rename(f"{pollutant_col}_2020plus_mean")
    comb = pd.concat([early_mean, late_mean], axis=1).reset_index()
    comb[f"{pollutant_col}_change_rate"] = (
        (comb[f"{pollutant_col}_2020plus_mean"] - comb[f"{pollutant_col}_2017_2019_mean"])
        / comb[f"{pollutant_col}_2017_2019_mean"]
        * 100
    )
    return comb


# 2017~2019 vs 2020~2023 구간 비교 (pm10, pm25 각각 계산)
pm10_cmp = compare_two_periods(pm10_daily, "pm10", (2017, 2019), (2020, 2023))
pm25_cmp = compare_two_periods(pm25_daily, "pm25", (2017, 2019), (2020, 2023))

# outer merge로 합친다. 전남·경북은 PM2.5 컬럼이 NaN.
period_compare = pd.merge(pm10_cmp, pm25_cmp, on="sido", how="outer")
period_compare = period_compare.sort_values("sido").reset_index(drop=True)

print("2017~2019 vs 2020~ 비교 테이블:", period_compare.shape)
period_compare.round(2)

2017~2019 vs 2020~ 비교 테이블: (17, 7)


,sido,pm10_2017_2019_mean,pm10_2020plus_mean,pm10_change_rate,pm25_2017_2019_mean,pm25_2020plus_mean,pm25_change_rate
0,강원,42.87,36.52,-14.83,28.38,20.25,-28.64
1,경기,55.26,47.48,-14.09,33.81,26.82,-20.68
2,경남,41.09,32.57,-20.75,21.74,18.08,-16.85
3,경북,45.88,39.21,-14.52,NaN,NaN,NaN
4,광주,45.53,38.91,-14.55,28.66,22.30,-22.17
5,대구,45.80,40.52,-11.53,28.30,22.33,-21.08
6,대전,51.43,42.54,-17.29,28.20,22.26,-21.06
7,부산,42.55,35.85,-15.73,25.37,19.99,-21.19
8,서울,50.98,44.41,-12.89,31.66,25.14,-20.59
9,세종,51.67,46.59,-9.84,29.73,25.64,-13.75


## 9. SAW 입력 데이터 생성

SAW 기준은 다음 세 가지다.

1. PM2.5 개선율
2. PM10 개선율
3. 시행 후 PM2.5 절대농도

개선 우선순위 관점에서는 정규화 방향이 중요하다.

- PM2.5 개선율: 낮을수록 우선순위 높음 → cost 기준
- PM10 개선율: 낮을수록 우선순위 높음 → cost 기준
- 시행 후 PM2.5 절대농도: 높을수록 우선순위 높음 → benefit 기준

> **⚠ 중요**: SAW 최종 점수 계산 시 PM2.5 관련 컬럼이 NaN인 시/도(전남·경북)는
> NaN으로 남아 자동으로 제외된다. 즉 SAW 대안은 PM2.5 기준으로 **15개** 시/도만 유효하다.

In [29]:
def normalize_benefit(series: pd.Series) -> pd.Series:
    """값이 클수록 좋은 기준을 0~1로 정규화한다."""
    min_value = series.min(skipna=True)
    max_value = series.max(skipna=True)
    return (series - min_value) / (max_value - min_value)


def normalize_cost(series: pd.Series) -> pd.Series:
    """값이 작을수록 좋은 기준을 0~1로 정규화한다."""
    min_value = series.min(skipna=True)
    max_value = series.max(skipna=True)
    return (max_value - series) / (max_value - min_value)


# SAW 입력 컬럼 추출 (전남·경북은 pm25 관련 컬럼이 NaN)
saw = improvement[["sido", "pm25_improvement_rate", "pm10_improvement_rate", "pm25_after"]].copy()
saw = saw.rename(columns={"pm25_after": "pm25_after_abs"})

# 개선율은 낮을수록 우선순위가 높으므로 cost 기준으로 정규화한다.
# NaN(전남·경북)은 skipna 처리가 되어 해당 행은 NaN으로 유지된다.
saw["pm25_improvement_priority"] = normalize_cost(saw["pm25_improvement_rate"])
saw["pm10_improvement_priority"] = normalize_cost(saw["pm10_improvement_rate"])

# 시행 후 PM2.5 절대농도는 높을수록 우선순위가 높으므로 benefit 기준으로 정규화한다.
saw["pm25_after_abs_priority"] = normalize_benefit(saw["pm25_after_abs"])

saw.round(3)

,sido,pm25_improvement_rate,pm10_improvement_rate,pm25_after_abs,pm25_improvement_priority,pm10_improvement_priority,pm25_after_abs_priority
0,세종,12.552,11.535,26.073,1.000,0.840,0.861
1,인천,17.523,16.178,25.187,0.788,0.627,0.781
2,서울,19.829,17.754,25.613,0.689,0.554,0.819
3,대구,21.011,17.517,22.930,0.639,0.565,0.577
4,충남,21.452,8.065,27.620,0.620,1.000,1.000
5,경남,23.396,29.790,18.210,0.537,0.000,0.152
6,울산,24.499,19.141,18.637,0.490,0.490,0.191
7,광주,25.687,20.711,22.246,0.439,0.418,0.516
8,충북,25.943,18.651,27.257,0.428,0.513,0.967
9,제주,26.001,15.627,16.521,0.426,0.652,0.000


## 10. 동일가중 SAW 점수 및 순위 계산

우선은 세 기준을 동일가중치로 계산한다.

이후 PM2.5를 더 중요하게 볼지, 절대농도를 더 중요하게 볼지에 따라 가중치를 바꿔 민감도 분석을 할 수 있다.
전남·경북은 PM2.5 관련 정규화 점수가 NaN이므로 SAW 최종 점수도 NaN으로 남아 순위에서 자동 제외된다.

In [30]:
weights = {
    "pm25_improvement_priority": 1 / 3,
    "pm10_improvement_priority": 1 / 3,
    "pm25_after_abs_priority":   1 / 3,
}

saw["saw_score_equal_weight"] = sum(
    saw[col] * w for col, w in weights.items()
)

# 전남·경북은 saw_score_equal_weight가 NaN이므로 rank에서 자동으로 제외된다.
saw["saw_rank_equal_weight"] = (
    saw["saw_score_equal_weight"].rank(ascending=False, method="min", na_option="bottom").astype("Int64")
)

saw = saw.sort_values("saw_rank_equal_weight").reset_index(drop=True)

print("SAW 최종 결과 (PM2.5 유효 15개 + 전남·경북 NaN):")
saw.round(3)

SAW 최종 결과 (PM2.5 유효 15개 + 전남·경북 NaN):


,sido,pm25_improvement_rate,pm10_improvement_rate,pm25_after_abs,pm25_improvement_priority,pm10_improvement_priority,pm25_after_abs_priority,saw_score_equal_weight,saw_rank_equal_weight
0,세종,12.552,11.535,26.073,1.000,0.840,0.861,0.900,1
1,충남,21.452,8.065,27.620,0.620,1.000,1.000,0.873,2
2,인천,17.523,16.178,25.187,0.788,0.627,0.781,0.732,3
3,서울,19.829,17.754,25.613,0.689,0.554,0.819,0.688,4
4,충북,25.943,18.651,27.257,0.428,0.513,0.967,0.636,5
5,대구,21.011,17.517,22.930,0.639,0.565,0.577,0.594,6
6,경기,29.268,21.333,27.278,0.286,0.389,0.969,0.548,7
7,광주,25.687,20.711,22.246,0.439,0.418,0.516,0.458,8
8,전북,30.124,23.180,24.810,0.250,0.304,0.747,0.434,9
9,대전,26.930,21.816,22.447,0.386,0.367,0.534,0.429,10


## 11. 결과 저장

아래 네 파일을 저장한다.

- `sido_winter_improvement.csv`: 시/도별 시행 전후 평균 및 개선율 (PM10 17개 + PM2.5 15개, outer merge)
- `sido_winter_bias_check.csv`: 측정소 수, 관측 일수, PM2.5 시작 연도 점검표
- `sido_saw_input_equal_weight.csv`: SAW 입력값, 정규화 점수, 동일가중 순위
- `sido_period_compare_2017_2020.csv`: 2017~2019 vs 2020~ 구간 비교 (신규)

In [31]:
improvement_path   = OUTPUT_DIR / "sido_winter_improvement.csv"
bias_path          = OUTPUT_DIR / "sido_winter_bias_check.csv"
saw_path           = OUTPUT_DIR / "sido_saw_input_equal_weight.csv"
compare_path       = OUTPUT_DIR / "sido_period_compare_2017_2020.csv"

# Excel에서 한글이 깨지지 않도록 utf-8-sig로 저장한다.
improvement.to_csv(improvement_path, index=False, encoding="utf-8-sig")
bias.to_csv(bias_path, index=False, encoding="utf-8-sig")
saw.to_csv(saw_path, index=False, encoding="utf-8-sig")
period_compare.to_csv(compare_path, index=False, encoding="utf-8-sig")

print("개선율 저장:", improvement_path)
print("편향 점검 저장:", bias_path)
print("SAW 입력 저장:", saw_path)
print("구간 비교 저장:", compare_path)

개선율 저장: ..\data\processed\sido_winter_improvement.csv
편향 점검 저장: ..\data\processed\sido_winter_bias_check.csv
SAW 입력 저장: ..\data\processed\sido_saw_input_equal_weight.csv
구간 비교 저장: ..\data\processed\sido_period_compare_2017_2020.csv


## 12. 해석 메모

### PM2.5 시/도 개수 축소 (17개 → 15개)
- 전남(유효 시작 2019-12), 경북(2020-04)은 "2018년 이전부터 유효 PM2.5 측정"이라는 기준에 미달하여 PM2.5 분석에서 제외된다.
- PM10은 두 지역 모두 정상이므로 17개 시/도 전부 포함.
- SAW 대안은 PM2.5 기준으로 **15개**만 유효하며, 전남·경북의 SAW 점수는 NaN으로 표시된다.

### before 구간 표본 부족 주의
- 경기·충남(2018-04 시작)은 before 구간(2015~2018 겨울) 중 2018 겨울 한 계절만 유효할 수 있다.
- 경남·울산·전북(2018-02 시작)도 유사한 상황이다.
- 표본이 1~2개인 경우 before 평균의 신뢰도가 낮으므로 개선율 해석 시 주의한다.

### 기타 해석 주의사항
- PM2.5 개선율이 낮고, 시행 후 PM2.5 절대농도가 높은 지역은 개선 우선순위가 높게 나온다.
- 단, 이 결과는 전후 비교 기반이다. 코로나19, 중국발 오염 변화, 산업 구조 변화, 측정망 변화 등 외부 요인을 분리한 인과효과는 아니다.
- SAW 가중치는 정책 판단에 따라 바뀔 수 있으므로 동일가중 결과만 최종 결론으로 고정하지 않는 것이 좋다.